# 🔗 End-to-End Supply Chain Analysis
### Data Analyst Portfolio Project 2026
**Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-Learn  
**Dataset:** Consumer Goods Supply Chain (Healthcare, Beauty, Skincare, Cosmetics)

---
## 📋 Business Problem
A consumer goods company wants to identify supply chain inefficiencies:
- Which product types drive revenue?
- Which suppliers cause delays and defects?
- Where are the delivery bottlenecks?
- Can we predict high-risk products using Machine Learning?


## 1️⃣ Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

## 2️⃣ Data Loading & First Look

In [ ]:
df = pd.read_csv("supply_chain_data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
# Missing values
df.isnull().sum()

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rev_by_cat = df.groupby("Product type")["Revenue generated"].sum().sort_values(ascending=False)
rev_by_cat.plot(kind="bar", ax=axes[0], color=sns.color_palette("muted", len(rev_by_cat)), edgecolor="white")
axes[0].set_title("Total Revenue by Product Type", fontsize=13, fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

sns.histplot(df["Profit margin"], bins=30, kde=True, ax=axes[1], color="#4C72B0")
axes[1].set_title("Profit Margin Distribution", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sold = df.groupby("Product type")["Number of products sold"].sum().sort_values(ascending=False)
sold.plot(kind="bar", ax=axes[0], color=sns.color_palette("pastel"), edgecolor="grey")
axes[0].set_title("Total Products Sold by Category", fontsize=13, fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

defect = df.groupby("Supplier name")["Defect rates"].mean().sort_values(ascending=False)
defect.plot(kind="bar", ax=axes[1], color=sns.color_palette("Set2"), edgecolor="grey")
axes[1].set_title("Avg Defect Rate by Supplier", fontsize=13, fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 4️⃣ Data Cleaning & Feature Engineering

In [ ]:
df["Order date"]     = pd.to_datetime(df["Order date"])
df["Shipping date"]  = pd.to_datetime(df["Shipping date"])
df["Delivery date"]  = pd.to_datetime(df["Delivery date"])

df["Processing days"]       = (df["Shipping date"] - df["Order date"]).dt.days
df["Transit days"]          = (df["Delivery date"] - df["Shipping date"]).dt.days
df["Total fulfillment days"]= (df["Delivery date"] - df["Order date"]).dt.days
df["On-time delivery"]      = (df["Total fulfillment days"] <= df["Lead time"]).astype(int)
df["Gross profit"]          = df["Revenue generated"] - df["Costs"]
df["Revenue per unit"]      = df["Revenue generated"] / df["Number of products sold"].replace(0, np.nan)
df["Order month"]           = df["Order date"].dt.to_period("M")
df["High defect"]           = (df["Defect rates"] > df["Defect rates"].median()).astype(int)
df["Risk flag"]             = ((df["High defect"] == 1) | (df["Inspection results"] == "Fail")).astype(int)
df["Est revenue loss"]      = df["Revenue generated"] * df["Defect rates"]

print(f"On-time delivery rate : {df['On-time delivery'].mean()*100:.1f}%")
print(f"High-risk products    : {df['Risk flag'].sum()} / {len(df)}")
df.head()

## 5️⃣ Research Questions

In [ ]:
# Q1: Revenue by product × location
rev_table = df.pivot_table(values="Revenue generated", index="Product type",
                           columns="Location", aggfunc="sum").fillna(0)
sns.heatmap(rev_table, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5)
plt.title("Revenue: Product × Location", fontsize=13, fontweight="bold")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# Q2: Transport cost by mode
trans_cost = df.groupby("Transportation modes")["Transportation costs"].mean().sort_values()
trans_cost.plot(kind="barh", color=sns.color_palette("coolwarm", 4))
plt.title("Avg Transport Cost by Mode"); plt.tight_layout(); plt.show()
print(trans_cost.round(2))

In [ ]:
# Q3: Customer segment revenue
seg_rev = df.groupby("Customer segment")["Revenue generated"].sum()
seg_rev.plot(kind="pie", autopct="%1.1f%%", startangle=90,
             colors=sns.color_palette("pastel", 3))
plt.title("Revenue by Customer Segment"); plt.ylabel(""); plt.show()

## 6️⃣ Bottleneck Detection

In [ ]:
print("Avg Processing Days:", df["Processing days"].mean().round(1))
print("Avg Transit Days   :", df["Transit days"].mean().round(1))
print("Avg Fulfillment Days:", df["Total fulfillment days"].mean().round(1))

bottleneck = df.groupby("Supplier name")["Total fulfillment days"].mean().sort_values(ascending=False)
bottleneck.plot(kind="bar", color="#E74C3C", edgecolor="white")
plt.axhline(df["Total fulfillment days"].mean(), color="navy", linestyle="--", label="Overall avg")
plt.title("Avg Fulfillment Days by Supplier", fontweight="bold")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# On-time vs Late scatter
colors = df["On-time delivery"].map({1:"#27AE60", 0:"#E74C3C"})
plt.scatter(df["Lead time"], df["Total fulfillment days"], c=colors, alpha=0.5, s=40)
plt.plot([0,30],[0,30],"k--",linewidth=1,label="Expected = Actual")
plt.title("Expected vs Actual Fulfillment Days", fontweight="bold")
plt.xlabel("Lead Time (Expected)"); plt.ylabel("Fulfillment Days (Actual)")
on_p  = mpatches.Patch(color="#27AE60", label="On-time")
late_p= mpatches.Patch(color="#E74C3C", label="Late")
plt.legend(handles=[on_p, late_p]); plt.tight_layout(); plt.show()

## 7️⃣ Root Cause Analysis

In [ ]:
sns.boxplot(data=df, x="Supplier name", y="Defect rates", palette="Set2")
plt.title("Defect Rate by Supplier", fontweight="bold")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

print("\nInspection Fail Rate by Supplier:")
print(df.groupby("Supplier name")["Inspection results"]
        .apply(lambda x: (x=="Fail").mean()*100).sort_values(ascending=False).round(1))

In [ ]:
loss = df.groupby("Product type")["Est revenue loss"].sum().sort_values(ascending=False)
loss.plot(kind="bar", color=sns.color_palette("Reds_r", len(loss)), edgecolor="white")
plt.title("Estimated Revenue Loss from Defects by Category", fontweight="bold")
plt.ylabel("₹ Revenue Loss"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 8️⃣ Time Series Analysis

In [ ]:
monthly = df.groupby("Order month").agg(
    Revenue=("Revenue generated","sum"),
    Orders=("SKU","count"),
    Defects=("Defect rates","mean"),
    AvgFulfillment=("Total fulfillment days","mean")
).reset_index()
monthly["Order month"] = monthly["Order month"].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes[0,0].plot(monthly["Order month"], monthly["Revenue"], marker="o", color="#2980B9", linewidth=2)
axes[0,0].set_title("Monthly Revenue"); axes[0,0].tick_params(axis="x", rotation=45)

axes[0,1].bar(monthly["Order month"], monthly["Orders"], color="#8E44AD", alpha=0.8)
axes[0,1].set_title("Monthly Orders"); axes[0,1].tick_params(axis="x", rotation=45)

axes[1,0].plot(monthly["Order month"], monthly["Defects"], marker="s", color="#E74C3C", linewidth=2)
axes[1,0].set_title("Avg Defect Rate"); axes[1,0].tick_params(axis="x", rotation=45)

axes[1,1].plot(monthly["Order month"], monthly["AvgFulfillment"], marker="^", color="#27AE60", linewidth=2)
axes[1,1].set_title("Avg Fulfillment Days"); axes[1,1].tick_params(axis="x", rotation=45)

plt.suptitle("Monthly Supply Chain Trends", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 9️⃣ Machine Learning – Risk Classification

In [ ]:
cat_cols = ["Product type","Supplier name","Transportation modes","Routes","Customer segment","Location"]
num_cols = ["Price","Availability","Number of products sold","Revenue generated",
            "Manufacturing costs","Costs","Profit margin","Order quantities",
            "Lead time","Processing days","Transit days","Transportation costs",
            "Production volumes","Stock levels","Manufacturing lead time"]

df_ml = df[cat_cols + num_cols + ["Risk flag"]].copy()
le = LabelEncoder()
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

X = df_ml.drop("Risk flag", axis=1)
y = df_ml["Risk flag"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

In [ ]:
models = {
    "Random Forest":     RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                      ("lr", LogisticRegression(max_iter=500, random_state=42))])
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    auc    = roc_auc_score(y_test, y_prob)
    results[name] = {"model":model,"y_pred":y_pred,"y_prob":y_prob,"auc":auc}
    print(f"\n{name}  AUC={auc:.3f}")
    print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
rf = results["Random Forest"]["model"]
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
fi.plot(kind="barh", color="#3498DB", edgecolor="white")
plt.title("Top Feature Importances (Random Forest)", fontweight="bold")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
# ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.2f})", linewidth=2)
plt.plot([0,1],[0,1],"k--"); plt.title("ROC Curves", fontweight="bold")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend(); plt.tight_layout(); plt.show()

## 🔟 Business Insights & Recommendations

| Area | Finding | Recommendation |
|---|---|---|
| 📦 Revenue | Beauty is the top category | Increase inventory & promotions for Beauty SKUs |
| 🏭 Suppliers | Supplier E has highest defect rate | Conduct quality audit; set SLA penalties |
| 🚚 Logistics | Road is most cost-effective | Use Road/Rail for non-urgent bulk shipments |
| ⚠️ Quality | 628 products flagged high-risk | Implement pre-shipment QC checkpoints |
| 📈 ML Model | Random Forest AUC ≈ 0.53 | Use model to flag risky SKUs before dispatch |
| 💸 Revenue Loss | ~₹3.85L lost to defects | Invest in defect-reduction programs with suppliers |
